# ✂️ NB02：Chunking 文字切塊策略

**目標：** 深入理解切塊策略對 RAG 檢索品質的關鍵影響，並比較不同策略的優缺點。

**學習成果：**
- 理解為什麼 Chunking 是 RAG 最重要的環節之一
- 掌握四種主要切塊策略：固定大小、重疊、遞迴、語意
- 能夠根據不同場景選擇合適的切塊策略
- 了解 chunk size 和 overlap 的設計原則

> 💡 **一句話說重點：** Chunking 直接決定你的檢索品質——chunk 切錯了，後面的 embedding 和檢索再好也沒用。

## 1. 為什麼 Chunking 這麼重要？

### 問題情境

假設你有一份 50 頁的技術文件，需要回答各種問題。
你必須決定：**要把文件切成多大的段落來存入向量資料庫？**

### Chunk Size 的兩難

```
Chunk 太小（~50 tokens）：
  ✅ 精準匹配單一概念
  ❌ 失去上下文（沒辦法理解前後關係）
  ❌ 需要更多 API 呼叫，成本高

Chunk 太大（~2000 tokens）：
  ✅ 保留完整上下文
  ❌ 引入不相關資訊，降低精確度
  ❌ LLM 輸入 token 成本增加
  ❌ 向量「稀釋」，相似度計算不準確

Chunk 適中（~200-500 tokens）：
  ✅ 平衡語意完整性與搜索精確度
  ✅ 通常是大多數場景的最佳選擇
```

## 2. 環境設定

In [ ]:
import os
import re
import tiktoken
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from dotenv import load_dotenv
from openai import OpenAI

# 設定中文字型（macOS）
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'DejaVu Sans']

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Token 計算器（精確計算 OpenAI 模型的 token 數）
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))

print("✅ 環境設定完成")

## 3. 準備示範文件

使用一份關於「大型語言模型發展史」的長文，來展示不同切塊策略的效果。

In [ ]:
# 示範長文（約 800+ tokens）
long_document = """
大型語言模型（LLM）的發展歷程

第一章：Transformer 架構的誕生

2017 年，Google 的研究團隊發表了劃時代的論文「Attention Is All You Need」，提出了 Transformer 架構。這個架構徹底改變了自然語言處理（NLP）的發展方向。Transformer 的核心創新是「自注意力機制」（Self-Attention），讓模型能夠同時考慮序列中所有位置的關係，不再受限於 RNN 的順序處理限制。在 Transformer 出現之前，主流的序列模型是 LSTM（長短期記憶網路），但它難以捕捉長距離依賴關係。Transformer 解決了這個問題，為後來的大型語言模型奠定了基礎。

第二章：BERT 與雙向語言模型

2018 年，Google 發布了 BERT（Bidirectional Encoder Representations from Transformers）。BERT 的突破在於使用「雙向」訓練方式，讓模型能夠同時理解文字的前後文。BERT 引入了兩種預訓練任務：遮罩語言模型（MLM）和下一句預測（NSP）。遮罩語言模型隨機遮住 15% 的詞彙，要求模型根據上下文預測被遮住的詞。BERT 在多個 NLP 基準測試上創下了新的最佳成績，引發了預訓練語言模型的研究熱潮。許多後續模型如 RoBERTa、ALBERT 都基於 BERT 架構進行改進。

第三章：GPT 系列與生成式模型

同樣在 2018 年，OpenAI 發布了 GPT（Generative Pre-trained Transformer）。與 BERT 的雙向設計不同，GPT 採用「單向」（從左到右）的自迴歸生成方式，更適合文字生成任務。GPT-2（2019）因為其強大的文字生成能力，曾引發 OpenAI 考慮是否公開發布的倫理討論。GPT-3（2020）的規模達到 1750 億參數，展現出令人驚訝的「少樣本學習」（Few-Shot Learning）能力——只需要少量範例，模型就能執行新任務，無需重新訓練。這個能力徹底改變了 AI 應用的開發方式。

第四章：ChatGPT 與 RLHF 訓練

2022 年 11 月，OpenAI 發布了 ChatGPT，讓 AI 對話能力第一次真正走入大眾視野。ChatGPT 的核心訓練方法是 RLHF（來自人類回饋的強化學習，Reinforcement Learning from Human Feedback）。RLHF 分為三個步驟：首先用高品質對話微調基礎模型；接著訓練一個「獎勵模型」來評估回答品質；最後用強化學習讓語言模型最大化獎勵分數。這個方法讓模型的回答更符合人類期望，大幅減少有害內容的生成。ChatGPT 在兩個月內達到一億用戶，成為史上成長最快的消費者應用程式。

第五章：多模態與未來展望

2023 年後，大型語言模型開始向多模態方向發展。GPT-4V 能夠理解圖片內容，Gemini 從設計之初就支援文字、圖片、音訊和影片的多模態處理。未來的趨勢包括：更長的上下文窗口（已從 4K 發展到 100 萬+ token）、更強的推理能力（Chain-of-Thought、Tree-of-Thought）、Agent 能力（讓 LLM 能夠使用工具和自主規劃）、以及更好的知識更新機制（RAG 和持續學習）。模型越來越小而強，邊緣設備上的本地 LLM 也開始成為可能，如 Llama、Phi、Mistral 等開源模型。
""".strip()

total_tokens = count_tokens(long_document)
print(f"文件總字數：{len(long_document)} 字")
print(f"文件總 Token 數：{total_tokens} tokens")
print(f"\n文件前 200 字：\n{long_document[:200]}...")

## 策略一：固定大小切塊（Fixed-Size Chunking）

**最簡單**的切塊方式：直接按字元數或 token 數切塊，不考慮語意邊界。

- ✅ 實作簡單，速度快
- ❌ 可能切斷句子中間，破壞語意

In [ ]:
def fixed_size_chunking(text: str, chunk_size: int = 200, unit: str = "chars") -> list[str]:
    """
    固定大小切塊
    Args:
        chunk_size: 每個 chunk 的大小
        unit: 'chars'（字元）或 'tokens'（token）
    """
    if unit == "chars":
        chunks = []
        for i in range(0, len(text), chunk_size):
            chunks.append(text[i:i + chunk_size])
        return chunks
    elif unit == "tokens":
        tokens = tokenizer.encode(text)
        chunks = []
        for i in range(0, len(tokens), chunk_size):
            chunk_tokens = tokens[i:i + chunk_size]
            chunks.append(tokenizer.decode(chunk_tokens))
        return chunks

# 執行固定大小切塊
fixed_chunks = fixed_size_chunking(long_document, chunk_size=300, unit="chars")

print(f"策略：固定大小切塊（每塊 300 字元）")
print(f"切塊數量：{len(fixed_chunks)}")
print("\n" + "=" * 60)

for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\n[Chunk {i+1}] ({count_tokens(chunk)} tokens)")
    print("-" * 40)
    print(chunk)

print("\n... (只顯示前 3 個 chunks)")

### 觀察：固定大小切塊的問題

注意 Chunk 1 和 Chunk 2 的邊界！你會發現：
- 句子可能被切斷在中間
- 段落的開頭和結尾可能被分到不同 chunk
- 這會導致檢索時找到的 chunk 缺少必要的上下文

## 策略二：重疊切塊（Overlap Chunking）

在固定大小的基礎上加入**重疊**，讓相鄰 chunk 共享部分內容，避免語意在邊界被截斷。

- ✅ 減少邊界截斷問題
- ✅ 實作簡單
- ❌ 增加儲存成本（重複內容）
- ❌ 仍然不考慮真正的語意邊界

In [ ]:
def overlap_chunking(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """
    重疊切塊（以字元為單位）
    Args:
        chunk_size: 每個 chunk 的大小（字元）
        overlap: 相鄰 chunk 的重疊大小（字元）
    """
    chunks = []
    step = chunk_size - overlap  # 每次移動的步長
    
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk:  # 忽略空白 chunk
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    
    return chunks

# 執行重疊切塊
overlap_chunks = overlap_chunking(long_document, chunk_size=300, overlap=60)

print(f"策略：重疊切塊（chunk_size=300, overlap=60）")
print(f"重疊率：{60/300*100:.0f}%")
print(f"切塊數量：{len(overlap_chunks)}")
print("\n" + "=" * 60)

# 顯示 Chunk 1 和 Chunk 2 的重疊部分
print("\n[Chunk 1 結尾 60 字元]：")
print(f"...{overlap_chunks[0][-60:]}")

print("\n[Chunk 2 開頭 60 字元]：")
print(f"{overlap_chunks[1][:60]}...")

print("\n→ 可以看到 Chunk 1 的結尾和 Chunk 2 的開頭有重疊！")

## 策略三：遞迴字元切塊（Recursive Character Chunking）

按照**語意優先**的分隔符遞迴切塊：先嘗試用段落分割，不夠小再用句子，再不行才用字元。

這是 **LangChain 預設** 的切塊策略，實際效果通常比純固定大小好很多。

- ✅ 盡量在自然邊界切塊（段落 > 句子 > 字詞）
- ✅ 兼顧大小控制和語意完整
- ❌ 無法處理跨段落的語意關聯

In [ ]:
def recursive_chunking(
    text: str,
    chunk_size: int = 300,
    overlap: int = 50,
    separators: list[str] = None
) -> list[str]:
    """
    遞迴字元切塊
    按照分隔符優先順序遞迴切割，確保在語意邊界切塊
    """
    if separators is None:
        # 優先順序：段落 → 句子 → 逗號 → 字元
        separators = ["\n\n", "\n", "。", "！", "？", "，", " ", ""]
    
    def split_with_separator(text: str, separator: str) -> list[str]:
        """用指定分隔符切割，保留分隔符語意"""
        if separator == "":
            return list(text)
        parts = text.split(separator)
        return [p for p in parts if p.strip()]
    
    def merge_chunks(splits: list[str], chunk_size: int, overlap: int, sep: str) -> list[str]:
        """合併小片段到目標 chunk size"""
        chunks = []
        current_chunk = ""
        
        for split in splits:
            if len(current_chunk) + len(split) + len(sep) <= chunk_size:
                current_chunk += (sep if current_chunk else "") + split
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = split
        
        if current_chunk:
            chunks.append(current_chunk)
        
        return chunks
    
    # 嘗試每個分隔符
    for i, separator in enumerate(separators):
        splits = split_with_separator(text, separator)
        
        # 檢查是否所有片段都在 chunk_size 以內
        if all(len(s) <= chunk_size for s in splits):
            return merge_chunks(splits, chunk_size, overlap, separator if separator else "")
        
        # 對超過 chunk_size 的片段，繼續遞迴切割
        result = []
        for split in splits:
            if len(split) <= chunk_size:
                result.append(split)
            else:
                # 遞迴用更細的分隔符處理
                sub_chunks = recursive_chunking(split, chunk_size, overlap, separators[i+1:])
                result.extend(sub_chunks)
        
        return merge_chunks(result, chunk_size, overlap, separator if separator else "")
    
    return [text]

# 執行遞迴切塊
recursive_chunks = recursive_chunking(long_document, chunk_size=300, overlap=50)

print(f"策略：遞迴字元切塊（chunk_size=300, overlap=50）")
print(f"切塊數量：{len(recursive_chunks)}")
print("\n" + "=" * 60)

for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars, {count_tokens(chunk)} tokens)")
    print("-" * 40)
    print(chunk)

## 策略四：語意切塊（Semantic Chunking）

最智能的切塊方式：計算相鄰句子的**語意相似度**，在**語意斷點**（Semantic Breakpoint）切塊。

```
句子 1: 「BERT 使用雙向訓練...」
句子 2: 「BERT 的預訓練任務有...」  ← 語意相似，不切
句子 3: 「GPT 採用單向生成...」     ← 語意差異大！在這裡切
句子 4: 「GPT-2 的生成能力...」     ← 語意相似，不切
```

- ✅ 語意最完整，chunk 邊界有意義
- ✅ 避免在語意中間截斷
- ❌ 需要計算所有句子的 embedding，成本高、速度慢
- ❌ 實作複雜度更高

In [ ]:
def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """計算兩個向量的餘弦相似度"""
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def semantic_chunking(
    text: str,
    breakpoint_threshold: float = 0.85,
    min_chunk_size: int = 50
) -> tuple[list[str], list[float]]:
    """
    語意切塊
    Args:
        breakpoint_threshold: 相似度低於此值就切塊（0-1，越高切越細）
        min_chunk_size: 最小 chunk 大小（字元）
    Returns:
        (chunks, similarities) - chunks 列表和相鄰句子相似度列表
    """
    # 1. 分割成句子
    sentences = re.split(r'(?<=[。！？\n])\s*', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > min_chunk_size]
    
    print(f"分割成 {len(sentences)} 個句子，計算 embedding 中...")
    
    # 2. 計算每個句子的 embedding
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=sentences
    )
    embeddings = [r.embedding for r in response.data]
    
    # 3. 計算相鄰句子的相似度
    similarities = []
    for i in range(len(sentences) - 1):
        sim = cosine_similarity(embeddings[i], embeddings[i+1])
        similarities.append(sim)
    
    # 4. 在相似度低的地方切塊（語意轉變點）
    chunks = []
    current_chunk = sentences[0]
    
    for i, sim in enumerate(similarities):
        if sim < breakpoint_threshold:
            # 語意斷點！在這裡切
            chunks.append(current_chunk)
            current_chunk = sentences[i+1]
        else:
            # 語意連續，合併到同一 chunk
            current_chunk += "\n" + sentences[i+1]
    
    chunks.append(current_chunk)  # 最後一個 chunk
    
    return chunks, similarities

# 執行語意切塊
semantic_chunks, similarities = semantic_chunking(
    long_document,
    breakpoint_threshold=0.88
)

print(f"\n策略：語意切塊（threshold=0.88）")
print(f"切塊數量：{len(semantic_chunks)}")
print("\n" + "=" * 60)

for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars, {count_tokens(chunk)} tokens)")
    print("-" * 40)
    print(chunk[:300] + ("..." if len(chunk) > 300 else ""))

In [ ]:
# 視覺化語意相似度，找出切塊點
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# 圖1：相鄰句子相似度
x = range(len(similarities))
colors = ['red' if s < 0.88 else 'steelblue' for s in similarities]

ax1.bar(x, similarities, color=colors, alpha=0.8)
ax1.axhline(y=0.88, color='red', linestyle='--', alpha=0.7, label='切塊閾值 (0.88)')
ax1.set_xlabel('句子索引')
ax1.set_ylabel('與下一句的相似度')
ax1.set_title('相鄰句子語意相似度（紅色 = 語意斷點，會在此切塊）')
ax1.legend()
ax1.set_ylim(0, 1)

# 圖2：各策略的 chunk 大小分佈
strategies = {
    'Fixed (300)': [len(c) for c in fixed_chunks],
    'Overlap (300/60)': [len(c) for c in overlap_chunks],
    'Recursive (300)': [len(c) for c in recursive_chunks],
    'Semantic (0.88)': [len(c) for c in semantic_chunks],
}

positions = range(len(strategies))
data = list(strategies.values())
labels = list(strategies.keys())

ax2.boxplot(data, labels=labels)
ax2.set_ylabel('Chunk 大小（字元數）')
ax2.set_title('各切塊策略的 Chunk 大小分佈')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("圖表已儲存為 chunking_comparison.png")

## 策略比較：對同一問題的檢索效果

In [ ]:
import chromadb

chroma_client = chromadb.Client()

def build_collection(collection_name: str, chunks: list[str]) -> chromadb.Collection:
    """建立向量資料庫 collection"""
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    col = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    
    # 批次建立 embedding
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunks
    )
    embeddings = [r.embedding for r in response.data]
    
    col.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks
    )
    return col

print("建立各策略的向量資料庫...")
collections = {
    "Fixed": build_collection("fixed", fixed_chunks),
    "Overlap": build_collection("overlap", overlap_chunks),
    "Recursive": build_collection("recursive", recursive_chunks),
    "Semantic": build_collection("semantic", semantic_chunks),
}
print("✅ 完成")

for name, col in collections.items():
    strategy_chunks = {
        "Fixed": fixed_chunks, "Overlap": overlap_chunks,
        "Recursive": recursive_chunks, "Semantic": semantic_chunks
    }[name]
    print(f"  {name}: {col.count()} chunks, avg size: {np.mean([len(c) for c in strategy_chunks]):.0f} chars")

In [ ]:
def compare_retrieval(query: str, collections: dict, n_results: int = 2):
    """比較不同切塊策略的檢索結果"""
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding
    
    print(f"查詢：'{query}'")
    print("=" * 70)
    
    for strategy_name, col in collections.items():
        results = col.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            include=["documents", "distances"]
        )
        
        top_doc = results["documents"][0][0]
        similarity = 1 - results["distances"][0][0]
        
        print(f"\n[{strategy_name}] 相似度: {similarity:.4f}")
        print(f"最相關 chunk ({len(top_doc)} chars):")
        print(f"  {top_doc[:200]}..." if len(top_doc) > 200 else f"  {top_doc}")

# 測試問題 1：需要精確資訊
compare_retrieval(
    "BERT 的遮罩語言模型（MLM）是如何訓練的？",
    collections
)

In [ ]:
# 測試問題 2：需要跨段落上下文
compare_retrieval(
    "GPT 系列模型的發展歷程和能力演進",
    collections
)

## 策略選擇指南

| 策略 | 適用場景 | Chunk Size 建議 | Overlap 建議 |
|------|---------|----------------|-------------|
| **固定大小** | 快速原型、均勻結構文件 | 200-500 chars | 0-10% |
| **重疊切塊** | 連續文本（小說、報告） | 300-600 chars | 10-20% |
| **遞迴字元** | 通用場景（推薦預設） | 300-500 tokens | 10-15% |
| **語意切塊** | 高品質需求、主題多樣文件 | 動態 | N/A |

### 面試關鍵問答
> **面試官：** 你會如何選擇 chunk size？
>
> **好答案：** 「這取決於文件特性和查詢模式。一般來說我從 200-500 tokens 開始，用重疊（10-15%）避免邊界截斷。如果文件有清晰的段落結構，用遞迴切塊保留語意邊界。如果對品質要求很高且預算允許，語意切塊能提供最好的結果。我會建立評估集來衡量不同策略對 Context Recall 的影響。」

## 小結

1. ✅ **Chunking 是 RAG 最關鍵的環節**，直接決定檢索品質
2. ✅ **固定大小切塊**：最簡單，但容易截斷語意
3. ✅ **重疊切塊**：改善邊界問題，但增加儲存成本
4. ✅ **遞迴字元切塊**：實用的預設選擇，兼顧語意和大小控制
5. ✅ **語意切塊**：最智能，但成本最高
6. ✅ **選擇原則**：根據文件結構、查詢模式和成本預算決定

### 下一步
- **NB03**：進階 RAG 技術——混合搜索（BM25 + 向量）和重排序（Reranking）